# Packages

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import json

import networkx as nx
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import plotly.graph_objects as go


sys.path.append(os.path.abspath(os.path.join('..')))

from src.random_graph import *
from src.plot import *
from src.process_data import *

# Pipeline

In [35]:
specie_list = [
    # 'GSD', 'HSC', 'mCAD', 'VSC', 
    'yeast','hESC', 'mESC', 
    'ecoli_gnw', 
    'human_trrust', 
    # "athaliana_wolf", "dmelanogaster_wolf", "hsapiens_wolf", "scerevisiae_wolf",
    "ecoli_wolf",
    'mDC'
    ]

properties_to_keep = [
    'graph_id',
    'avg_path_dir', 'diameter',
    'nb_edges', 
    'Cascade',
    'FFL', 'Fan-In', 'Fan-Out',
    'Mutual-In', 'Mutual-Out',
    'Regulating-Mutual'
]

idx_list = {}
idces_list = {}

for specie in specie_list:
    # Load ground truth
    input_dir = f"../../data/graphs/"
    ground_truth_graph, ground_truth_stat, ground_truth_motifs = load_graphs(input_dir+specie)
    ground_truth_all_stats = ground_truth_stat | ground_truth_motifs
    
    # Load random data from Parquet
    output_dir = f"../../data/random_graphs/{specie}"
    parquet_path = f'{output_dir}/{specie}_results.parquet'
    df_random = pd.read_parquet(parquet_path, columns=properties_to_keep)
    
    # Computing errors
    gt_prop = {
        key: value 
        for key, value in ground_truth_all_stats.items()
        if key in properties_to_keep
    }
    
    # Computing errors
    df_errors = compute_relative_error(df_random, gt_prop)
    df_errors.to_csv(f"{output_dir}/relative_error.csv", index=False)
    
    # Select best graph
    idx_list[specie] = get_best_index(df_errors)
    idces_list[specie] = get_best_indices(df_errors, n_top=5)
    
    print(f'Best graph index for dataset {specie}: {idx_list[specie]}')
    
    ################## Plot relative histogram ########################
    # Prepare data
    df = df_errors.drop(['total_error', 'graph_id'], axis=1)

    # Define labels
    label_mapping = {
        'avg_path_dir': "Average Path Length", 
        'diameter': "Diameter",
        'nb_edges': "Arc", 
        'Cascade': "Cascade", 
        'FFL': "Feed Forward Loop", 
        'Fan-In': "Fan-In",
        'Fan-Out': "Fan-Out", 
        'Mutual-In': "Mutual-In",
        'Mutual-Out': "Mutual-Out",
        'Regulating-Mutual': "Regulating-Mutual",
    }


    fig = plot_histogram_distribution(
        df,
        label_mapping=label_mapping,
        n_bins=40,
        save_path=f"{output_dir}/properties_distribution_{specie}.pdf"
    )
    fig.write_image(f"{output_dir}/properties_distribution_{specie}.svg")
    fig.write_image(f"{output_dir}/properties_distribution_{specie}.png")
    # fig.show()

    ############# Plot best profiles ##############
    n_top=5
    smallest_list = list(
        df_errors["total_error"]
        .abs()
        .nsmallest(n_top)
        .index
    )
    df = df_errors.loc[smallest_list].drop(['total_error','graph_id'], axis = 1)
    df_best = df_errors.loc[idces_list[specie]].drop(['total_error','graph_id'], axis = 1)
    df_best = df_best.rename(columns=label_mapping)

    params = list(df_best.columns)

    fig = go.Figure()

    # Samples
    legend_label=['1st', '2nd', '3rd', '4th', '5th']
    for i, (idx, row) in enumerate(df_best.iterrows(), start=1):
        fig.add_trace(go.Scatter(
            x=params,
            y=row[params].tolist(),
            mode='lines+markers',
            name=f"{legend_label[i-1]}",
            opacity=0.5,
            hovertemplate="Sample %{text}<br>%{x}: %{y:.4f}",
            text=[idx] * len(params)
        ))
    fig.update_layout(
        font=dict(
            family="Times New Roman, serif",
            size=20,
            color="black"
        ),
        legend=dict(
        title=dict(text="<b>Top 5 GRN</b>", font=dict(size=18)),
        bordercolor="Black",
        borderwidth=1,
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02 # This places the legend outside the plot to the right
    ),
        yaxis_title=f"Relative Difference",
        hovermode="x unified",
        margin=dict(
            t=10, 
            b=100,
            l=80,
            r=250
        ),
        width=1200,
        height=400,
        yaxis_range=[-1, 1],
        xaxis=dict(tickangle=25)
    )
    fig.write_image(f"{output_dir}/5_profile_relative_{specie}.pdf")
    fig.write_image(f"{output_dir}/5_profile_relative_{specie}.svg")
    fig.write_image(f"{output_dir}/5_profile_relative_{specie}.png")
    # fig.show()
    
    plot_deg_ref_vs_multi_sim(
            ground_truth_graph, 
            parquet_path,
            ref_label=specie,
            # save_path=f"{output_dir}/degree_distribution_{specie}.pdf", 
        )

Loaded graph from ../../data/graphs/yeast_g.graphml.
Loaded graph stats from ../../data/graphs/yeast_stat.json.
Loaded graph motif counts from ../../data/graphs/yeast_motifs.json.
Best graph index for dataset yeast: 889


/var/tmp/ipykernel_67674/321828473.py:80: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:81: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:140: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:141: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be re

Loaded graph from ../../data/graphs/hESC_g.graphml.
Loaded graph stats from ../../data/graphs/hESC_stat.json.
Loaded graph motif counts from ../../data/graphs/hESC_motifs.json.
Best graph index for dataset hESC: 271


/var/tmp/ipykernel_67674/321828473.py:80: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:81: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:140: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:141: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be re

Loaded graph from ../../data/graphs/mESC_g.graphml.
Loaded graph stats from ../../data/graphs/mESC_stat.json.
Loaded graph motif counts from ../../data/graphs/mESC_motifs.json.
Best graph index for dataset mESC: 938


...

/var/tmp/ipykernel_67674/321828473.py:80: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:81: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:140: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:141: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be re

Loaded graph from ../../data/graphs/ecoli_wolf_g.graphml.
Loaded graph stats from ../../data/graphs/ecoli_wolf_stat.json.
Loaded graph motif counts from ../../data/graphs/ecoli_wolf_motifs.json.
Best graph index for dataset ecoli_wolf: 785


/var/tmp/ipykernel_67674/321828473.py:80: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:81: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:140: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:141: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be re

Loaded graph from ../../data/graphs/mDC_g.graphml.
Loaded graph stats from ../../data/graphs/mDC_stat.json.
Loaded graph motif counts from ../../data/graphs/mDC_motifs.json.
Best graph index for dataset mDC: 190


/var/tmp/ipykernel_67674/321828473.py:80: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:81: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:140: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/var/tmp/ipykernel_67674/321828473.py:141: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be re

# End